# NBA Draft Value Analysis (2000–2020)
### Who Were the Biggest Steals and Busts?

This notebook analyzes **21 years of NBA draft data** to answer:
- What career value does each draft position *typically* produce?
- Which players massively **outperformed** their draft slot (steals)?
- Which high picks **underperformed** expectations (busts)?
- Which **teams** are the best and worst drafters?
- Can we **predict** career value from draft position?

**Data:** Basketball Reference (scraped)  
**Value metric:** Composite of Win Shares + VORP  
**Models:** Polynomial regression (baseline), Gradient Boosting (improved)

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

from scraper    import scrape_all_drafts
from preprocess import clean_and_engineer
from analysis   import run_all_analyses
from model      import run_models
from visualize  import generate_all_figures, set_style

set_style()
print('Setup complete.')

## 1. Scrape Data

This scrapes Basketball Reference for all NBA drafts 2000–2020.  
**Takes ~5 minutes** due to rate limiting (4s per request, ~21 requests).  
Data is saved to `data/draft_data_raw.csv` — skip scraping if file exists.

In [ ]:
RAW_PATH = '../data/draft_data_raw.csv'

if os.path.exists(RAW_PATH):
    print('Loading cached data...')
    raw_df = pd.read_csv(RAW_PATH)
    print(f'Loaded {len(raw_df)} picks from cache.')
else:
    print('Scraping data from Basketball Reference...')
    raw_df = scrape_all_drafts(start=2000, end=2020, save_path=RAW_PATH)

raw_df.head()

## 2. Clean & Engineer Features

In [ ]:
df = clean_and_engineer(raw_df)
print(f'\nDataset: {len(df)} picks across {df["draft_year"].nunique()} draft years')
df[['player','draft_year','pick','pick_tier','win_shares','VORP','value_score','value_over_expected']].head(10)

In [ ]:
# Distribution of value scores
print('Value Score Distribution:')
df['value_score'].describe().round(2)

## 3. Run All Analyses

In [ ]:
results       = run_all_analyses(df)
model_results = run_models(df)

## 4. Draft Value Curve

In [ ]:
print('Pick Value Curve (median WS by pick):')
results['pick_value_curve'].head(15)

In [ ]:
print(f'Baseline model (pick → WS): MAE={model_results["baseline"]["cv_mae"]} WS, R²={model_results["baseline"]["cv_r2"]}')
print(f'Improved model (pick+era → value): MAE={model_results["improved"]["cv_mae"]}, R²={model_results["improved"]["cv_r2"]}')
print(f'\nFeature Importances:')
print(model_results['improved']['feature_importance'])

## 5. Greatest Steals

In [ ]:
print('TOP 20 DRAFT STEALS (2000-2020)')
print('Players who most outperformed their draft position expected value')
print('='*70)
results['top_steals'][['player','draft_year','pick','win_shares','VORP','value_over_expected']]

## 6. Biggest Busts

In [ ]:
print('TOP 20 DRAFT BUSTS (2000-2020, Picks 1-20 only)')
print('High picks who most underperformed their expected value')
print('='*70)
results['top_busts'][['player','draft_year','pick','win_shares','VORP','value_over_expected']]

## 7. Best Second-Round Picks

In [ ]:
print('BEST SECOND ROUND PICKS (2000-2020)')
results['second_round_steals'][['player','draft_year','pick','win_shares','VORP']]

## 8. Value by Pick Tier

In [ ]:
results['value_by_tier'].style \
    .background_gradient(cmap='Blues', subset=['median_ws']) \
    .background_gradient(cmap='Reds',  subset=['pct_busts'])

## 9. Team Drafting Efficiency

In [ ]:
print('BEST DRAFTING TEAMS (min 10 picks, 2000-2020)')
model_results['teams'][['team','total_picks','avg_value_over_exp','total_value_over_exp','best_pick']].head(15)

## 10. Generate All Figures

In [ ]:
figure_paths = generate_all_figures(results, model_results)

for path in figure_paths:
    img = mpimg.imread(path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 11. Key Findings

*(Fill in after running)*

### Draft Value Curve
- Picks 1–5 produce a median of ~X Win Shares; this drops sharply after pick ~10.
- The model explains ~X% of variance in career value from pick number alone.

### Greatest Steals
- The biggest steal was **[Player]** (pick #X, year), who produced X Win Shares above expectation.
- Second round steals like **[Player]** prove late-round value is real.

### Biggest Busts
- The biggest bust was **[Player]** (pick #X), who produced X WS below expectation for a top-20 pick.

### Team Efficiency
- **[Team]** had the highest average value over expected, consistently finding value late.
- **[Team]** was the worst drafter, repeatedly missing on high picks.